# Analyze Study (Binary Classification)

This notebook walks through the post-hoc analysis of a completed Octopus classification study. It covers study validation, performance evaluation, feature selection summaries, and feature importance analysis.

Each section can be run independently once the study is loaded.

**Prerequisite:** Run `basic_classification.py` first to generate the study data:

```bash
python examples/basic_classification.py
```

In [ ]:
from octopus.analysis import (
    aucroc_plot,
    confusion_matrix_plot,
    feature_count_plot,
    feature_frequency_plot,
    fi_plot,
    get_details,
    load_study_info,
    performance,
    performance_plot,
    performance_table,
    selected_features,
    workflow_graph,
)

## Load Study

Provide the path to your study directory. If you pass a name prefix (e.g. `"../studies/my_study"`), `load_study_info` automatically finds the latest timestamped run matching that prefix. The study directory is validated on load -- missing outersplits or workflow task directories raise an error immediately.


In [ ]:
study_info = load_study_info("../studies/basic_classification")

## Study Details

`get_details` returns a summary of the study configuration (ML type, target metric, number of folds, and available outer split IDs).

`workflow_graph` displays the workflow tasks and their dependencies as a tree. Each node shows the task ID, module type, and description. Branches indicate that multiple tasks depend on the same parent.


In [ ]:
details = get_details(study_info)
details

In [ ]:
workflow = workflow_graph(study_info)
print(workflow)

## Performance

`performance` returns metric scores per outer split for all prediction tasks in the workflow. Feature selection tasks are skipped automatically. By default the `dev` partition is shown -- use this partition for model selection and hyperparameter decisions to avoid data leakage from the held-out test set. Only use `partition="test"` for final, unbiased evaluation after all decisions have been made. Use `metric` to filter to a single metric, or `task` to select a specific workflow task.

**How to read the plot:**

- Each bar represents the metric score for one outer split. Consistent bar heights across splits indicate stable model performance.
- Large variation between splits may signal that the model is sensitive to the particular train/test partition, which can happen with small datasets or class imbalance.
- When multiple tasks are shown, comparing bars across tasks reveals whether later workflow steps (e.g. after feature selection) improve or degrade performance.


In [ ]:
perf = performance(study_info)
fig = performance_plot(perf)
fig.show()

### Performance Table

`performance_table` shows the train and dev scores for a single task and metric side by side, with one row per outer split and a Mean row. Comparing train and dev columns reveals whether the model is overfitting (large gap) or underfitting (both scores poor).


In [ ]:
performance_table(study_info, task=0, metric="AUCROC")

## Selected Features

`selected_features` reads `selected_features.json` from each outersplit and task directory and returns two tables:

- **feature_table**: number of selected features per outersplit and task (plus a Mean row).
- **frequency_table**: how often each feature was selected across outersplits, sorted descending.

**How to read the plots:**

- The **feature count plot** shows how many features were selected in each outer split (and task). A consistent count across splits suggests a stable feature selection. A large drop between workflow tasks confirms that feature selection steps are reducing dimensionality as intended.
- The **feature frequency plot** shows which features are most consistently selected across outer splits. Features selected in all splits are robust and likely genuinely informative. Features selected in only one or two splits may be noise or split-specific artefacts.


In [ ]:
feature_table, frequency_table = selected_features(study_info)
fig = feature_count_plot(feature_table)
fig.show()

In [ ]:
fig = feature_frequency_plot(frequency_table)
fig.show()

## Test Set Performance

The sections above use the **dev** partition -- these results should guide model selection, hyperparameter tuning, and feature selection decisions. Looking at test scores during these steps introduces data leakage and inflates reported performance.

The **test** partition below provides an unbiased estimate of final model performance. Only look at test results **after all modelling decisions have been made**. If test results lead you to change the model or features, the test set is no longer independent and the reported scores lose their validity.

`OctoTestEvaluator` re-computes metrics on the held-out test data per outersplit.


In [ ]:
from octopus.analysis import OctoTestEvaluator

task_predictor_test = OctoTestEvaluator(study=study_info, task_id=0, result_type="best")

## AUCROC Curves

`aucroc_plot` returns a ROC (Receiver Operating Characteristic) curve computed on the held-out test data. The ROC curve plots the True Positive Rate (sensitivity) against the False Positive Rate (1 - specificity) at different classification thresholds. The Area Under the Curve (AUC) summarises the model's ability to distinguish between classes into a single number.

**How to read the plot:**

- A curve hugging the top-left corner indicates strong discrimination between classes (AUC close to 1.0).
- The dashed diagonal represents random guessing (AUC = 0.5). Any model should be well above this line.
- An AUC above 0.8 is generally considered good, above 0.9 excellent, though this depends on the domain.
- In the averaged plot, a narrow standard deviation band suggests consistent performance across splits, while a wide band indicates that model performance varies depending on how the data is split.

**Available modes** (via the `mode` parameter):

- **`"merged"`** (default): Pools predictions from all outersplits into a single curve. Gives one overall view of discriminative performance.
- **`"averaged"`**: Computes a ROC curve per outersplit, then plots the mean with a +/- 1 standard deviation band. Shows how stable the model performs across different data splits.
- **`"per_split"`**: Shows one subplot per outersplit side by side. Useful for inspecting individual split performance.


In [ ]:
fig = aucroc_plot(task_predictor_test, mode="merged")
fig.show()

## Confusion Matrix

`confusion_matrix_plot` shows a confusion matrix for each outer split as a subplot. Rows represent the true class labels, columns represent the predicted class labels. Each cell shows how many (or what percentage of) samples with a given true label were assigned to each predicted label.

**How to read the plot:**

- Diagonal cells (top-left to bottom-right) are correct predictions. High values on the diagonal mean the model classifies well.
- Off-diagonal cells are misclassifications. For example, a high value in row "A", column "B" means the model frequently predicts class B when the true class is A.
- In `"relative"` mode, each row sums to 100%. This makes it easier to compare classes with different sample sizes -- a row showing 95% on the diagonal means the model correctly identifies 95% of that class (recall).

**Available modes** (via the `mode` parameter):

- **`"count"`** (default): Absolute counts of predicted vs. true class labels.
- **`"relative"`**: Row-normalised percentages (recall per true class).


In [ ]:
fig = confusion_matrix_plot(task_predictor_test, mode="relative")
fig.show()

## Feature Importance

Feature importance measures how much each feature contributes to the model's predictions. `calculate_fi` computes importance on the held-out test data using the final fitted models. Each outer-split model is evaluated independently on its own test fold, then results are aggregated into an ensemble summary.

Two methods are available:

- **Permutation importance** (`fi_type="permutation"`): Randomly shuffles each feature and measures the drop in model performance. A large drop means the feature is important. Use `"group_permutation"` to also compute importance at the feature-group level. Fast and model-agnostic.
- **SHAP** (`fi_type="shap"`): Computes Shapley values that attribute the prediction to individual features. More fine-grained than permutation but slower. Available `shap_type` options: `"kernel"` (default, model-agnostic), `"permutation"`, `"exact"` (slowest, most accurate).

**How to read the plot:**

- Features are sorted by importance. The top features contribute most to the model's predictions.
- Error bars (when available) show confidence intervals across outer splits. Narrow bars indicate consistent importance, wide bars suggest the feature's contribution varies by split.
- Features with importance near zero or negative can often be removed without hurting performance.


### Permutation Feature Importance

`n_repeats` controls how many times each feature is shuffled. Higher values give more stable importance estimates and tighter confidence intervals. For this example we use `n_repeats=3` to keep runtime short. For real analyses, `n_repeats=10` (default) or higher is recommended.


In [ ]:
fi_table_perm = task_predictor_test.calculate_fi(fi_type="permutation", n_repeats=3)
fig = fi_plot(fi_table_perm)
fig.show()

### SHAP Feature Importance


In [ ]:
fi_table_shap = task_predictor_test.calculate_fi(fi_type="shap", shap_type="kernel")
fig = fi_plot(fi_table_shap)
fig.show()